This notebook takes the icdar training data and generates a csv file with writer,same_text,isEng,train,file_name,male columns (file_name is the absolute path)

In [4]:
#imports
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import random

# Set the random seed for reproducibility
seed=42
np.random.seed(seed)

data_PATH = "C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset"
image_PATH=data_PATH+"\\unzipped"
output_path=".\\datasets"
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [5]:
###### Main functions #########

def get_df_from_folders(image_PATH, folder_names):
    # Loop through each directory and collect image file paths for labeled images only
    image_dirs = [os.path.join(image_PATH, folder) for folder in folder_names]
    writers = []
    isEng = []
    same_text = []
    file_names = []

    for image_dir in image_dirs:
        for f in os.listdir(image_dir):
            if f.endswith('.jpg'):
                base_name = os.path.splitext(f)[0]  # Remove extension
                parts = base_name.split('_')

                if len(parts) != 2:
                    continue  # Skip files that don't follow the expected pattern

                index, version = parts

                if int(version)>2:
                    isEng.append(1)
                else:
                    isEng.append(0)
                if int(version)%2==0:
                    same_text.append(1)
                else:
                    same_text.append(0)
                file_names.append(os.path.join(image_dir,f))
                writers.append(int(index))

    # Create a dataframe from the extracted index and version values
    train_file_df = pd.DataFrame({'writer': writers, 'isEng': isEng, 'same_text': same_text,'file_name':file_names})
    return train_file_df


##### Checking functions #####

def male_counts(sex_df):
    # Get the counts of each unique value in the "male" column
    male_counts = sex_df['male'].value_counts(dropna=False)

    # Print the counts
    print("Number of times 'male' is 0:", male_counts.get(0, 0))
    print("Number of times 'male' is 1:", male_counts.get(1, 0))
    print("Number of times 'male' is something else:", len(sex_df) - male_counts.get(0, 0) - male_counts.get(1, 0))
def check_if_both(train_df, column_name='same_text'):
    # Group by writer and check if both same_text=1 and same_text=0 are present
    writer_groups = train_df.groupby('writer')[column_name].nunique()

    # Filter writers that do not have both same_text=1 and same_text=0
    writers_missing_both = writer_groups[writer_groups != 2]

    if writers_missing_both.empty:
        print(f"All writers have both {column_name}=1 and {column_name}=0.")
    else:
        print(f"The following writers do not have {column_name}=1 and {column_name}=0")
        print(writers_missing_both)
def check_randomization(train_df):
    # Get the number of rows where train == 1
    train_1_count = train_df[train_df['train'] == 1].shape[0]

    # Calculate the fraction
    train_1_fraction = train_1_count / train_df.shape[0]

    print(f"Number of rows where train == 1: {train_1_count}")
    print(f"Fraction of rows where train == 1: {train_1_fraction:.2f}")
def check_grouping(train_df):
    # Group by writer and check if the train column has a constant value
    constant_train_check = train_df.groupby('writer')['train'].nunique()

    # Find writers where the train column is not constant
    non_constant_writers = constant_train_check[constant_train_check > 1]

    if non_constant_writers.empty:
        print("The train column is constant for all writers.")
    else:
        print("The train column is not constant for the following writers:")
        print(non_constant_writers)
def check_occurrences(train_df):
    # Count the occurrences of each unique writer value
    writer_counts = train_df['writer'].value_counts()

    # Check if all writers have exactly 4 occurrences
    if (writer_counts == 4).all():
        print("Each unique writer value occurs on exactly 4 rows.")
    else:
        print("Some writers do not occur exactly 4 times.")
        print(writer_counts[writer_counts != 4])
def check_title_association(train_df):
    random_numbers = random.sample(range(1, 282*4+1), 10)
    for n in random_numbers:
        print(n)
        print(train_df['file_name'][n])
        print(train_df['writer'][n],train_df['isEng'][n], train_df['same_text'][n])
        print('-------------')
def check_sex_association(train_df,sex_df):
    random_numbers = random.sample(range(1, 283), 10)
    for n in random_numbers:
        print(n)
        print(train_df[train_df['writer'] == n][['writer','male']])
        print(sex_df[sex_df['writer'] == n][['writer','male']])
        print('-------------')
def check_if_seed(train_df):
    train_0_writers = train_df[train_df['train'] == 0]['writer'].unique().tolist()
    train_1_writers = train_df[train_df['train'] == 1]['writer'].unique().tolist()
    return train_0_writers, train_1_writers


###### Saving functions #######

import json

def get_base_metadata(filepath):
    stats = os.stat(filepath)
    return {
        "full_path": os.path.abspath(filepath),
        "size_bytes": stats.st_size,
        "created": datetime.fromtimestamp(stats.st_ctime).isoformat(),
        "modified": datetime.fromtimestamp(stats.st_mtime).isoformat(),
        "accessed": datetime.fromtimestamp(stats.st_atime).isoformat()
    }

def load_log(path):
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    return {}

def save_log(data, path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=4)

def add_or_update_file(filepath, log_path, custom_metadata=None):
    """
    Adds or updates a file's metadata entry, including custom metadata.
    """
    if not os.path.isfile(filepath):
        print(f"File not found: {filepath}")
        return
    
    filename = os.path.basename(filepath)
    log = load_log(log_path)

    base_meta = get_base_metadata(filepath)
    entry = log.get(filename, {})

    # Combine existing metadata, new base, and new custom metadata
    entry.update(base_meta)
    if custom_metadata:
        entry.update(custom_metadata)

    log[filename] = entry
    save_log(log, log_path)
    print(f"Updated log for {filename}")

def read_metadata(filepath, log_path):
    """
    Adds or updates a file's metadata entry, including custom metadata.
    """
    if not os.path.isfile(filepath):
        print(f"File not found: {filepath}")
        return
    
    filename = os.path.basename(filepath)
    log = load_log(log_path)

    entry = log.get(filename, None)
    if entry:
        print(f"Metadata for {filename}:")
        for key, value in entry.items():
            print(f"{key}: {value}")
    else:
        print(f"No metadata found for {filename}")



In [10]:
sex_df = pd.read_csv(os.path.join(data_PATH, "train_answers.csv"),delimiter=',')
sex_df_test = pd.read_csv(os.path.join(data_PATH, "test_answers.csv"),delimiter=',')
private=sex_df_test[sex_df_test['Usage']=='PrivateTest']
public=sex_df_test[sex_df_test['Usage']=='PublicTest']

N=282
# Create a dataframe with writer column from 1 to 282
writers_df = pd.DataFrame({'writer': np.arange(1, N+1)})

#get folders with the train data
def is_valid_folder(folder):
    parts = folder.split('_')
    if len(parts) == 0:
        return False
    try:
        int(parts[0])
        return True
    except ValueError:
        return False
folder_names = [folder for folder in os.listdir(image_PATH) if os.path.isdir(os.path.join(image_PATH, folder)) and is_valid_folder(folder)]
# Extract the X part from the folder names
x_values = [int(folder.split('_')[0]) for folder in folder_names]
# Sort both lists based on the X values
sorted_indices = sorted(range(len(x_values)), key=lambda k: x_values[k])
folder_names = [folder_names[i] for i in sorted_indices]
x_values = [x_values[i] for i in sorted_indices]
print(folder_names)

train_file_df = get_df_from_folders(image_PATH, folder_names)

train_df = train_file_df[train_file_df['writer']<=N].copy()
train_df = train_df.merge(sex_df, on=['writer'], how='left')
train_df = train_df.merge(writers_df, on=['writer'], how='left')
train_df['index'] = train_df.index

test_df = train_file_df[train_file_df['writer']>N].copy()
test_df = test_df.merge(sex_df_test, on=['writer'], how='left')

public_df = test_df[test_df['Usage']=='PublicTest']
private_df = test_df[test_df['Usage']=='PrivateTest']

#perform checks
for df,name in zip([train_df, public_df, private_df], ['train', 'public', 'private']):
    print(f"Checking {name} dataframe with {len(df)} rows")
    male_counts(df)
    check_if_both(df, column_name='same_text')
    check_if_both(df, column_name='isEng')
    #check_randomization(df)
    #check_grouping(df)
    check_occurrences(df)

    print("Saving dataframe to disk")
    df.to_csv(os.path.join(output_path, f"icdar_{name}_df.csv"), index=False)


['1_50', '51_100', '101_150', '151_200', '201_250', '251_300', '301_350', '351_400', '401_450', '451_475']
Checking train dataframe with 1128 rows
Number of times 'male' is 0: 572
Number of times 'male' is 1: 556
Number of times 'male' is something else: 0
All writers have both same_text=1 and same_text=0.
All writers have both isEng=1 and isEng=0.
Each unique writer value occurs on exactly 4 rows.
Saving dataframe to disk
Checking public dataframe with 284 rows
Number of times 'male' is 0: 172
Number of times 'male' is 1: 112
Number of times 'male' is something else: 0
All writers have both same_text=1 and same_text=0.
All writers have both isEng=1 and isEng=0.
Each unique writer value occurs on exactly 4 rows.
Saving dataframe to disk
Checking private dataframe with 488 rows
Number of times 'male' is 0: 272
Number of times 'male' is 1: 216
Number of times 'male' is something else: 0
All writers have both same_text=1 and same_text=0.
All writers have both isEng=1 and isEng=0.
Each uni